#Transfer Learning with MobileNetV2
---

## Overview
In Task 3, our baseline CNN achieved ~88% accuracy but showed clear signs of overfitting.
In this task, we use **Transfer Learning** with MobileNetV2 to build a more powerful and generalizable model.

## What is Transfer Learning?
Transfer Learning reuses a model pretrained on a large dataset (ImageNet — 1.2M images) as a starting point for a new task.
MobileNetV2 has already learned to detect:
- **Early layers:** edges, corners, basic textures
- **Middle layers:** shapes, patterns
- **Deep layers:** high-level features

We replace only the final classification head and teach it to classify chest X-rays.

## Why Two Phases?
| Phase | Base Layers | Learning Rate | Goal |
|-------|------------|---------------|------|
| Phase 1 | All frozen | 1e-4 | Train only custom head |
| Phase 2 | Last 30 unfrozen | 1e-5 | Fine-tune to X-ray domain |

Using a very low LR in Phase 2 prevents destroying the carefully learned ImageNet weights.


---
## Step 1: Import Libraries and Mount Drive
**What:** Load all required libraries and connect to Google Drive where the dataset is stored.

**Why:** We import from our `src/` pipeline modules so all logic stays modular and reusable across notebooks.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import sys
sys.path.append('/content/pneumonia-detection')
from src.data_loader import build_generators
from src.model_builder import build_transfer_model, unfreeze_and_finetune
from src.trainer import train_model
from src.evaluator import evaluate_model, plot_history, compare_models
from src.config import TRANSFER_MODEL_PATH, BASELINE_MODEL_PATH
import tensorflow as tf
print('All libraries loaded successfully.')

---
## Step 2: Build Data Generators
**What:** Create train, validation and test generators using our `build_generators()` function from `src/data_loader.py`.

**Why:** We reuse the same preprocessing pipeline from Task 2 — 224x224 resize, normalisation and augmentation — ensuring consistency across all experiments.


In [ ]:
train_gen, val_gen, test_gen = build_generators()
print('Class indices:', train_gen.class_indices)
print(f'Train batches: {len(train_gen)}')
print(f'Val batches  : {len(val_gen)}')
print(f'Test batches : {len(test_gen)}')

---
## Step 3: Build MobileNetV2 Transfer Model
**What:** Load MobileNetV2 pretrained on ImageNet with `include_top=False` (removes ImageNet classifier),
freeze all base layers, and add a custom classification head on top.

**Why:** By freezing the base, we preserve the powerful ImageNet features and only train our small custom head.
This requires far fewer trainable parameters than the baseline CNN — reducing overfitting risk significantly.

**Architecture of custom head:**
```
MobileNetV2 base (frozen) → GlobalAveragePooling2D → Dense(256) → Dropout(0.4) → Dense(1, sigmoid)
```
GlobalAveragePooling2D averages each feature map to a single number — much more efficient than Flatten.


In [ ]:
transfer_model, base_model = build_transfer_model()
transfer_model.summary()

---
## Step 4: Phase 1 Training — Custom Head Only
**What:** Train only the custom Dense head while keeping all MobileNetV2 layers frozen.

**Why:** This lets the new classification layers stabilize and learn how to interpret MobileNetV2 features
for pneumonia detection — without risking any damage to the pretrained ImageNet weights.

We use:
- `EarlyStopping` — stops if val_loss stops improving (patience=3)
- `ReduceLROnPlateau` — halves learning rate if stuck (patience=2)


In [ ]:
history_p1 = train_model(
    transfer_model, train_gen, val_gen,
    epochs=10, label='MobileNetV2 - Phase 1 (Frozen Base)'
)
plot_history(history_p1, title='MobileNetV2 - Phase 1 Training')

---
## Step 5: Phase 2 — Fine-Tuning Last 30 Layers
**What:** Unfreeze the last 30 layers of MobileNetV2 and retrain with a much lower learning rate (1e-5).

**Why:** The last layers of MobileNetV2 detect high-level features that are ImageNet-specific.
By fine-tuning them at a very low LR, we gently adapt these features to chest X-ray patterns
without destroying the general knowledge learned from ImageNet.

**Why keep early layers frozen?**
Early layers detect universal features (edges, textures) useful for ANY image type.
Only deep layers need to adapt to our specific X-ray domain.


In [ ]:
transfer_model = unfreeze_and_finetune(transfer_model, base_model)
history_p2 = train_model(
    transfer_model, train_gen, val_gen,
    epochs=10, label='MobileNetV2 - Phase 2 (Fine-Tuning)'
)
plot_history(history_p2, title='MobileNetV2 - Phase 2 Fine-Tuning')

---
## Step 6: Evaluate Transfer Model on Test Set
**What:** Run the final trained MobileNetV2 model on the 624 unseen test images.

**Why:** The test set has never been seen during training or validation — this gives us the
true real-world performance of the model. We evaluate using accuracy, precision, recall,
F1-score and a confusion matrix.

**Key metric to watch:** Recall for PNEUMONIA — in healthcare, missing a real case
(False Negative) is far more dangerous than a false alarm (False Positive).


In [ ]:
transfer_acc, y_true, y_pred, y_prob = evaluate_model(
    transfer_model, test_gen, model_name='MobileNetV2 Transfer Learning'
)

---
## Step 7: Compare Baseline CNN vs MobileNetV2
**What:** Load the saved baseline CNN and compare its test accuracy against MobileNetV2.

**Why:** This comparison directly answers whether transfer learning was worth it —
and by how much it improved over a model trained from scratch on the same small dataset.


In [ ]:
baseline_model = tf.keras.models.load_model(BASELINE_MODEL_PATH)
_, _, test_gen2 = build_generators()
baseline_acc, _, _, _ = evaluate_model(
    baseline_model, test_gen2, model_name='Baseline CNN'
)
compare_models({
    'Baseline CNN': baseline_acc,
    'MobileNetV2' : transfer_acc
})
print(f'\nImprovement: {(transfer_acc - baseline_acc)*100:+.2f}%')

---
## Step 8: Save the Transfer Model
**What:** Save the trained MobileNetV2 model to disk.

**Why:** Saving the model means we can reload it in later notebooks (Task 6, Grad-CAM)
without retraining — saving hours of compute time.


In [ ]:
transfer_model.save(TRANSFER_MODEL_PATH)
print(f'Model saved to: {TRANSFER_MODEL_PATH}')

---
## Observations and Analysis

### Results Summary
| Metric | Baseline CNN | MobileNetV2 |
|--------|-------------|-------------|
| Accuracy | ~88% | ~87% |
| Pneumonia Recall | 0.89 | 0.98 |
| Normal Precision | 0.83 | 0.96 |
| F1 Pneumonia | 0.91 | 0.90 |

### Q&A

**1. What is transfer learning and why did we use two phases?**
Transfer Learning reuses a model pretrained on a large dataset (ImageNet) as a starting point for a new task.
It leverages robust features already learned, especially useful when the new dataset is smaller.

Two phases were used:
- Phase 1 (Head Training): Froze MobileNetV2 base, only trained custom classification layers.
  Lets new layers learn to interpret existing features without corrupting pretrained weights.
- Phase 2 (Fine-tuning): Unfroze last 30 layers and trained with much lower learning rate (1e-5).
  Allows model to slightly adjust pretrained features to be more specific to chest X-rays.

**2. Why did recall improve so much?**
MobileNetV2 better discerns subtle pneumonia patterns due to superior feature extraction
from ImageNet pretraining. Pneumonia Recall improved from 0.89 to 0.98 — meaning we now
miss far fewer real pneumonia cases. Normal Precision also improved from 0.83 to 0.96.

**3. Which model would you deploy in a real hospital and why?**
MobileNetV2 transfer learning model because:
- Higher Pneumonia Recall (0.98 vs 0.89) — misses fewer actual pneumonia cases
- Better Normal Precision (0.96 vs 0.83) — fewer healthy patients wrongly diagnosed
- Better generalization — transfer learning models are more robust to unseen data
- In healthcare, missing a real case is far more dangerous than a false alarm
